# T05 CNN FER2013

Mô hình sáu lớp tích chập theo ví dụ của thầy. RandomAffine và RandomHorizontalFlip thay ImageDataGenerator trong tập train. Tập validation và test chỉ resize/chuyển tensor. Lần chạy trên ảnh thư mục sử dụng manifest BTL02 để tránh trùng pixel giữa các tập.

Chạy từ môi trường cài requirements. Đổi cấu hình và tên lượt chạy nếu muốn thí nghiệm mới. Notebook dùng kết quả đã lưu nếu lượt chạy đã hoàn tất.

In [ ]:
from pathlib import Path
import json
import sys

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
if not (root / "src" / "train_cnn.py").is_file():
    raise RuntimeError("Mở notebook từ thư mục BTL03 hoặc notebooks")
sys.path.insert(0, str(root))
from src.train_cnn import train_cnn
import os
os.chdir(root)

local_data = Path("data/fer2013")
local_ready = (local_data / "fer2013_training_onehot.csv").is_file() or (local_data / "train").is_dir()
image_dir = Path("../BTL02_Emotion_Recognition/data/raw")
manifest = Path("../BTL02_Emotion_Recognition/results/training/baseline_seed42/split_manifest.csv")
configs = {"fer2013": dict(
    data_dir=local_data if local_ready or not image_dir.is_dir() else image_dir,
    split_manifest=manifest if not local_ready and image_dir.is_dir() and manifest.is_file() else None,
    epochs=12,
)}
run_suffix = "cpu_seed42"

In [ ]:
runs = {}
for dataset, config in configs.items():
    run = Path("results") / f"{dataset}_{run_suffix}"
    if not (run / "metrics.json").is_file():
        train_cnn(dataset, output_dir=run, **config)
    runs[dataset] = run
    print(json.loads((run / "metrics.json").read_text(encoding="utf-8")))

In [ ]:
import csv
import matplotlib.pyplot as plt

for dataset, run in runs.items():
    with (run / "history.csv").open() as file:
        history = list(csv.DictReader(file))
    fig, axes = plt.subplots(1, 2, figsize=(9, 3))
    for metric, ax in zip(("loss", "accuracy"), axes):
        for split in ("train", "val"):
            ax.plot([int(row["epoch"]) for row in history],
                    [float(row[f"{split}_{metric}"]) for row in history],
                    marker="o", label=split)
        ax.set(xlabel="Epoch", ylabel=metric, title=dataset)
        ax.legend()
    fig.tight_layout()
    fig.savefig(run / "learning_curve.png", dpi=160)
    plt.show()